In [5]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types
from PIL import Image

# ----------------------------
# Load environment variables
# ----------------------------
load_dotenv()

PROJECT_ID = "challengegemini"
LOCATION   = os.environ.get("GOOGLE_CLOUD_LOCATION", "us-central1")

# ----------------------------
# Inputs
# ----------------------------
PROMPT = "Generate a cinematic portrait. Preserve the subject identity from the reference image."

ASPECT_RATIO = "3:4"     # Options: "1:1", "3:4", "4:3", "16:9"
RESOLUTION   = "2K"      # Options: "512px", "1K", "2K", "4K"

# Local reference image paths
REF_IMAGE_PATHS = [
    "ref_2.png",   # put your image file here
    # "ref_3.png", # you can add more (up to ~14)
]

# ----------------------------
# Load reference images
# ----------------------------
ref_images = []
for path in REF_IMAGE_PATHS:
    img = Image.open(path).convert("RGBA")
    ref_images.append(img)

# ----------------------------
# Create Vertex AI GenAI client
# ----------------------------
client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION,
)

# ----------------------------
# Generate image
# ----------------------------
response = client.models.generate_content(
    model="gemini-3.1-flash-image-preview",
    contents=[PROMPT, *ref_images],
    config=types.GenerateContentConfig(
        response_modalities=["TEXT", "IMAGE"],
        image_config=types.ImageConfig(
            aspect_ratio=ASPECT_RATIO,
            image_size=RESOLUTION,
        ),
    ),
)

# ----------------------------
# Save generated image
# ----------------------------
saved = False

for part in response.parts:
    img = part.as_image()
    if img is not None:
        img.save("nanobanana_out.png")
        saved = True
        break

print("Saved:", saved, "-> nanobanana_out.png")

INFO: The user provided project/location will take precedence over the Vertex AI API key from the environment variable.


INFO: AFC is enabled with max remote calls: 10.
INFO: HTTP Request: POST https://aiplatform.googleapis.com/v1beta1/projects/challengegemini/locations/global/publishers/google/models/gemini-3.1-flash-image-preview:generateContent "HTTP/1.1 200 OK"


Saved: True -> nanobanana_out.png


# video with reference 

In [15]:
import os
import time
import uuid
from urllib.parse import urlparse

from dotenv import load_dotenv
from google import genai
from google.genai import types
from google.cloud import storage

# ----------------------------
# Env / config
# ----------------------------
load_dotenv()

PROJECT_ID = "challengegemini"
LOCATION = os.environ.get("GOOGLE_CLOUD_LOCATION", "us-central1")

# REQUIRED: a bucket you can write to (same creds as your Vertex calls)
# Put this in .env: VEO_BUCKET=your-bucket-name
# VEO_BUCKET = os.environ["VEO_BUCKET"]
VEO_BUCKET = "challengegemini-storyteller"

# Inputs
LOCAL_REF_IMAGE = "nanobanana_out.png"  # <-- your Nano Banana output image
PROMPT = "Generate a short cinematic clip that keeps the character consistent with the reference image."

MODEL = os.environ.get("VEO_MODEL", "veo-3.1-fast-generate-001")  # you can change if needed
ASPECT_RATIO = "16:9"  # "16:9" or "9:16"
DURATION_SECONDS = 8   # 8/6/4 depending on availability
RESOLUTION = "720p"    # 720p/1080p/4k depending on availability
GENERATE_AUDIO = False
SEED = 12345

# ----------------------------
# Helpers
# ----------------------------
def upload_file_to_gcs(local_path: str, bucket_name: str, dest_blob: str) -> str:
    client = storage.Client(project=PROJECT_ID)
    bucket = client.bucket(bucket_name)

    blob = bucket.blob(dest_blob)
    # Set content-type based on extension
    content_type = "image/png" if local_path.lower().endswith(".png") else "image/jpeg"
    blob.upload_from_filename(local_path, content_type=content_type)
    return f"gs://{bucket_name}/{dest_blob}"

def parse_gs_uri(gs_uri: str):
    u = urlparse(gs_uri)
    if u.scheme != "gs":
        raise ValueError(f"Not a gs:// uri: {gs_uri}")
    return u.netloc, u.path.lstrip("/")

def download_gcs_to_file(gs_uri: str, local_path: str):
    bucket_name, blob_name = parse_gs_uri(gs_uri)
    client = storage.Client(project=PROJECT_ID)
    client.bucket(bucket_name).blob(blob_name).download_to_filename(local_path)

def wait_for_op(client: genai.Client, op, poll_s: int = 15, timeout_s: int = 1800):
    start = time.time()
    while not op.done:
        if time.time() - start > timeout_s:
            raise TimeoutError(f"Timed out after {timeout_s}s waiting for Veo operation")
        time.sleep(poll_s)
        op = client.operations.get(op)
    return op

# ----------------------------
# 1) Upload local reference image to GCS
# ----------------------------
run_id = uuid.uuid4().hex[:8]
ref_blob = f"veo_inputs/{run_id}/nanobanana_out.png"
ref_gcs_uri = upload_file_to_gcs(LOCAL_REF_IMAGE, VEO_BUCKET, ref_blob)
print("Uploaded reference image ->", ref_gcs_uri)

# Output prefix in GCS (Veo writes here)
output_gcs_uri = f"gs://{VEO_BUCKET}/veo_outputs/{run_id}/"

# ----------------------------
# 2) Call Veo (Vertex AI)
# ----------------------------
client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

operation = client.models.generate_videos(
    model=MODEL,
    prompt=PROMPT,
    config=types.GenerateVideosConfig(
        reference_images=[
            types.VideoGenerationReferenceImage(
                image=types.Image(
                    gcs_uri=ref_gcs_uri,
                    mime_type="image/png",
                ),
                reference_type="asset",
            )
        ],
        aspect_ratio=ASPECT_RATIO,
        duration_seconds=DURATION_SECONDS,
        resolution=RESOLUTION,
        generate_audio=GENERATE_AUDIO,
        seed=SEED,
        output_gcs_uri=output_gcs_uri,
    ),
)

# ----------------------------
# 3) Poll until done
# ----------------------------
operation = wait_for_op(client, operation, poll_s=15, timeout_s=1800)

if not operation.response:
    raise RuntimeError(f"Veo operation finished but returned no response. Full op: {operation}")

result = operation.result
video_gcs_uri = result.generated_videos[0].video.uri
print("Video stored on GCS ->", video_gcs_uri)

# ----------------------------
# 4) Download the mp4 locally
# ----------------------------
local_mp4 = f"veo_{run_id}.mp4"
download_gcs_to_file(video_gcs_uri, local_mp4)
print("Downloaded ->", local_mp4)

INFO: The user provided project/location will take precedence over the Vertex AI API key from the environment variable.


Uploaded reference image -> gs://challengegemini-storyteller/veo_inputs/75cf659d/nanobanana_out.png


INFO: HTTP Request: POST https://aiplatform.googleapis.com/v1beta1/projects/challengegemini/locations/global/publishers/google/models/veo-3.1-fast-generate-001:predictLongRunning "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://aiplatform.googleapis.com/v1beta1/projects/challengegemini/locations/global/publishers/google/models/veo-3.1-fast-generate-001:fetchPredictOperation "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://aiplatform.googleapis.com/v1beta1/projects/challengegemini/locations/global/publishers/google/models/veo-3.1-fast-generate-001:fetchPredictOperation "HTTP/1.1 200 OK"


Video stored on GCS -> gs://challengegemini-storyteller/veo_outputs/75cf659d/7173435769584321610/sample_0.mp4
Downloaded -> veo_75cf659d.mp4


In [10]:
import google.auth
creds, proj = google.auth.default()
print("ADC project:", proj)
print("Creds type:", type(creds))
print("Service account:", getattr(creds, "service_account_email", None))

ADC project: challengegemini-veo
Creds type: <class 'google.oauth2.service_account.Credentials'>
Service account: storyteller@challengegemini.iam.gserviceaccount.com
